### Edge

#### 1、条件分支 conditional_edge

In [3]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
    extra_body={
        "thinking": {"type": "disabled"}
}
)

In [ ]:
from typing import TypedDict, Literal, Sequence
from langgraph.graph import StateGraph, START, END
from IPython.display import display

class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    story: str
    content_type: str

def poem_node(state: OverAllState) -> OverAllState:
    response = model.invoke(f"创作一个主题是{state['topic']}的诗歌")
    return {
        "poem": response.content
    }

def joke_node(state: OverAllState) -> OverAllState:
    response = model.invoke(f"创作一个主题是{state['topic']}的笑话")
    return {
        "joke": response.content
    }

def story_node(state: OverAllState) -> OverAllState:
    response = model.invoke(f"创作一个主题是{state['topic']}的故事")
    return {
        "story": response.content
    }

def my_router(state: OverAllState) -> Sequence[Literal["joke", "story", "poem"]]:
    if state["content_type"] == "relax":
        return ["joke", "story"]
    else:
        return ["poem"]

builder = StateGraph(OverAllState)
builder.add_node(poem_node)
builder.add_node(joke_node)
builder.add_node(story_node)


builder.add_conditional_edges(START, my_router, path_map={
    "joke": "joke_node",
    "story": "story_node",
    "poem": "poem_node",
})
builder.add_edge("poem_node", END)
builder.add_edge("joke_node", END)
builder.add_edge("story_node", END)

graph = builder.compile()

result = graph.invoke(
    {
        "topic": "暹罗猫",
        "content_type": "relax",
    }
)

rprint(result)

display(graph)

#### 2、动态分支 dynamic branch

##### 2.1 Send

In [ ]:
from typing import TypedDict, Literal, Sequence
from langgraph.graph import StateGraph, START, END
from IPython.display import display

